# Introducing strides

In [54]:
import numpy as np
x = np.arange(8,dtype=np.int8)
x

array([0, 1, 2, 3, 4, 5, 6, 7], dtype=int8)

In [55]:
x.strides

(1,)

In [56]:
str(x.data)

'<memory at 0x000001CC19302E00>'

In [57]:
x.shape = 2,4 
x

array([[0, 1, 2, 3],
       [4, 5, 6, 7]], dtype=int8)

In [58]:
x.strides

(4, 1)

In [59]:
x.tobytes()

b'\x00\x01\x02\x03\x04\x05\x06\x07'

In [60]:
x.shape = 1, 4, 2
print(x.strides)
print(x.tobytes())

(8, 2, 1)
b'\x00\x01\x02\x03\x04\x05\x06\x07'


In [61]:
x = np.ones((10000,))
y = np.ones((10000*100,))[::100]
x.shape, y.shape

((10000,), (10000,))

In [62]:
x == y

array([ True,  True,  True, ...,  True,  True,  True])

In [63]:
print(x.flags)
print(y.flags)

  C_CONTIGUOUS : True
  F_CONTIGUOUS : True
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

  C_CONTIGUOUS : False
  F_CONTIGUOUS : False
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False



In [64]:
x.strides, y.strides

((8,), (800,))

In [65]:
%timeit x.sum()
%timeit y.sum() # y has larger strides so the summation if much slower

3.99 μs ± 50.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
7.81 μs ± 67.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


# Structured Array

In [66]:
x = np.empty((2,), dtype = ('i4,f4,S10'))

In [67]:
x[:] = [(1,1,'NumPy'),(10,-0.5,"Essential")]
x # 'f0', 'f1', 'f2' are default field name 

array([( 1,  1. , b'NumPy'), (10, -0.5, b'Essential')],
      dtype=[('f0', '<i4'), ('f1', '<f4'), ('f2', 'S10')])

In [68]:
x[0]

np.void((1, 1.0, b'NumPy'), dtype=[('f0', '<i4'), ('f1', '<f4'), ('f2', 'S10')])

In [69]:
x['f2']

array([b'NumPy', b'Essential'], dtype='|S10')

In [70]:
y = x['f0']
y

array([ 1, 10], dtype=int32)

In [71]:
y[:] = y * 10
y

array([ 10, 100], dtype=int32)

In [72]:
x

array([( 10,  1. , b'NumPy'), (100, -0.5, b'Essential')],
      dtype=[('f0', '<i4'), ('f1', '<f4'), ('f2', 'S10')])

In [73]:
z = np.ones((2,), dtype = ('3i4,(2,3)f4'))
z

array([([1, 1, 1], [[1., 1., 1.], [1., 1., 1.]]),
       ([1, 1, 1], [[1., 1., 1.], [1., 1., 1.]])],
      dtype=[('f0', '<i4', (3,)), ('f1', '<f4', (2, 3))])

In [74]:
x.dtype.names

('f0', 'f1', 'f2')

In [75]:
x.dtype.names = ('id','value','note')
x

array([( 10,  1. , b'NumPy'), (100, -0.5, b'Essential')],
      dtype=[('id', '<i4'), ('value', '<f4'), ('note', 'S10')])

In [76]:
list_ex = np.zeros((2,), dtype = [('id','i4'),('value','f4'),('note','S10')])
list_ex

array([(0, 0., b''), (0, 0., b'')],
      dtype=[('id', '<i4'), ('value', '<f4'), ('note', 'S10')])

In [77]:
dtype_field = ['id','value','note']
dtype_formats = ['i4','f4','S10']
list_ex = np.zeros((2,),dtype = {'names':dtype_field,'formats':dtype_formats})
list_ex

array([(0, 0., b''), (0, 0., b'')],
      dtype=[('id', '<i4'), ('value', '<f4'), ('note', 'S10')])

In [78]:
x[dtype_field[:2]]

array([( 10,  1. ), (100, -0.5)],
      dtype={'names': ['id', 'value'], 'formats': ['<i4', '<f4'], 'offsets': [0, 4], 'itemsize': 18})

In [79]:
import numpy as np

names = ['location','population','percent','date','source']
formats = ['U32','i8','f4','U16','U32']

dtype = np.dtype({
    'names': names,
    'formats': formats
})

data = [
    ('China', 1412000000, 0.177, '2024', 'UN'),
    ('India', 1393000000, 0.175, '2024', 'UN'),
    ('USA', 331000000, 0.042, '2024', 'UN'),
    ('Indonesia', 273000000, 0.034, '2024', 'UN'),
]

arr = np.array(data, dtype=dtype)

print(arr)

[('China', 1412000000, 0.177, '2024', 'UN')
 ('India', 1393000000, 0.175, '2024', 'UN')
 ('USA',  331000000, 0.042, '2024', 'UN')
 ('Indonesia',  273000000, 0.034, '2024', 'UN')]


In [80]:
x = np.datetime64('2015-04-01')
y = np.datetime64('2015-04')
x.dtype, y.dtype

(dtype('<M8[D]'), dtype('<M8[M]'))

In [81]:
np.datetime64('2015-04-01') - np.datetime64('2015-04')

np.timedelta64(0,'D')

In [82]:
np.datetime64('2015') + np.timedelta64(3, 'Y')

np.datetime64('2018')

# File I/O Submodule

In [83]:
ids = np.arange(1000)
values = np.random.random(1000)
days = np.random.randint(0, 365, 1000) * np.timedelta64(1,'D')
dates = np.datetime64('2014-01-01') + days

dtype = np.dtype([
    ('id', 'i4'),
    ('value', 'f4'),
    ('date', 'datetime64[D]')
])

rec_array = np.rec.fromarrays([ids, values, dates], dtype = dtype)
rec_array[:5]



rec.array([(0, 0.47918648, '2014-05-06'), (1, 0.9652084 , '2014-04-08'),
           (2, 0.99610865, '2014-10-12'), (3, 0.06894616, '2014-05-17'),
           (4, 0.54552627, '2014-10-20')],
          dtype=[('id', '<i4'), ('value', '<f4'), ('date', '<M8[D]')])

In [84]:
np.save('data.npy', rec_array)

In [85]:
# load data
data = np.load('data.npy', allow_pickle = True)
print(data[:5])

[(0, 0.47918648, '2014-05-06') (1, 0.9652084 , '2014-04-08')
 (2, 0.99610865, '2014-10-12') (3, 0.06894616, '2014-05-17')
 (4, 0.54552627, '2014-10-20')]


In [86]:
np.savez('data.npz', rec_array=rec_array)

In [87]:
data = np.load('data.npz')
rec_array = data['rec_array']
print(rec_array[:5])

[(0, 0.47918648, '2014-05-06') (1, 0.9652084 , '2014-04-08')
 (2, 0.99610865, '2014-10-12') (3, 0.06894616, '2014-05-17')
 (4, 0.54552627, '2014-10-20')]


In [90]:
np.savetxt(
    'data.csv',
    rec_array,
    delimiter=',',
    fmt=['%d', '%.6f', '%s'],  # 注意 date 要用字符串
    header='id,value,date',
    comments=''
)

In [ ]:
data = np.genfromtxt(
    'data.csv',
    delimiter=',',
    names=True,
    dtype=None,
    encoding=None
)

print(data[:5])

[(0, 0.166917, '2014-11-15') (1, 0.804423, '2014-03-25')
 (2, 0.976929, '2014-05-19') (3, 0.397736, '2014-11-29')
 (4, 0.87707 , '2014-05-20')]


In [ ]:
data['date'] = data['date'].astype('datetime64[D]')
print(data[:5])

[(0, 0.166917, '2014-11-15') (1, 0.804423, '2014-03-25')
 (2, 0.976929, '2014-05-19') (3, 0.397736, '2014-11-29')
 (4, 0.87707 , '2014-05-20')]


In [92]:
import pandas as pd

df = pd.DataFrame(rec_array)

In [ ]:
print(df)

      id     value       date
0      0  0.479186 2014-05-06
1      1  0.965208 2014-04-08
2      2  0.996109 2014-10-12
3      3  0.068946 2014-05-17
4      4  0.545526 2014-10-20
..   ...       ...        ...
995  995  0.043652 2014-02-06
996  996  0.691727 2014-03-04
997  997  0.822072 2014-12-12
998  998  0.488210 2014-07-13
999  999  0.929047 2014-04-30

[1000 rows x 3 columns]


In [95]:
mask = rec_array['value'] >= 0.75
print(mask[:5])

[False  True  True False False]


In [103]:
data = np.load('data.npz')
rec_array = data['rec_array']
high_value_rows = rec_array[rec_array['value'] >= 0.75]
mask = rec_array['value'] >= 0.75

from numpy.lib.recfunctions import append_fields
arr2 = append_fields(rec_array, 'mask', mask, dtypes = 'i1')
arr2 = np.asarray(arr2)
arr2[:5]

array([(0, 0.47918648, '2014-05-06', 0), (1, 0.9652084 , '2014-04-08', 1),
       (2, 0.99610865, '2014-10-12', 1), (3, 0.06894616, '2014-05-17', 0),
       (4, 0.54552627, '2014-10-20', 0)],
      dtype=[('id', '<i4'), ('value', '<f4'), ('date', '<M8[D]'), ('mask', 'i1')])

In [104]:
np.asarray?

Docstring:
asarray(a, dtype=None, order=None, *, device=None, copy=None, like=None)

Convert the input to an array.

Parameters
----------
a : array_like
    Input data, in any form that can be converted to an array.  This
    includes lists, lists of tuples, tuples, tuples of tuples, tuples
    of lists and ndarrays.
dtype : data-type, optional
    By default, the data-type is inferred from the input data.
order : {'C', 'F', 'A', 'K'}, optional
    Memory layout.  'A' and 'K' depend on the order of input array a.
    'C' row-major (C-style),
    'F' column-major (Fortran-style) memory representation.
    'A' (any) means 'F' if `a` is Fortran contiguous, 'C' otherwise
    'K' (keep) preserve input order
    Defaults to 'K'.
device : str, optional
    The device on which to place the created array. Default: ``None``.
    For Array-API interoperability only, so must be ``"cpu"`` if passed.

    .. versionadded:: 2.0.0
copy : bool, optional
    If ``True``, then the object is copied. If `